# Relative Frequency of 'et' and 'in' (Ausonius, Vergil, Cento)

Counts are taken from the masked corpora (`lexical_model/`), which preserve function
words while masking content words. Fisher exact tests use the one-sided alternative
chosen from the observed frequencies (always `less` for these two words).

In [1]:
from __future__ import annotations

from pathlib import Path

from scipy.stats import fisher_exact

BASE = Path("lexical_model")
WORDS = ["et", "in"]


def load_files(directory: Path) -> list[Path]:
    return sorted(directory.glob("*_sentences.quesplit.masked.txt"))


def count_word(files: list[Path], word: str) -> tuple[int, int]:
    total = 0
    count = 0
    for path in files:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                tokens = line.strip().split()
                total += len(tokens)
                count += sum(1 for t in tokens if t == word)
    return total, count

In [2]:
ausonius_files = load_files(BASE / "ausonius")
georgics_files = load_files(BASE / "vergil")
aeneid_files = load_files(BASE / "aeneid")
cento_files = load_files(BASE / "question")

corpora = {
    "Ausonius": ausonius_files,
    "Georgics": georgics_files,
    "Aeneid": aeneid_files,
    "Georgics+Aeneid": georgics_files + aeneid_files,
    "Cento (question)": cento_files,
}

stats = {word: {} for word in WORDS}
for word in WORDS:
    print(f"{word}: ")
    for name, files in corpora.items():
        total, count = count_word(files, word)
        freq = count / total if total else 0.0
        stats[word][name] = (total, count, freq)
        print(f"  {name:<18} tokens={total:>7,}  count={count:>5,}  freq={freq*100:.3f}%")

et: 
  Ausonius           tokens= 17,776  count=  387  freq=2.177%
  Georgics           tokens= 17,802  count=  567  freq=3.185%
  Aeneid             tokens= 81,140  count=2,157  freq=2.658%
  Georgics+Aeneid    tokens= 98,942  count=2,724  freq=2.753%
  Cento (question)   tokens=  1,079  count=   23  freq=2.132%
in: 
  Ausonius           tokens= 17,776  count=  148  freq=0.833%
  Georgics           tokens= 17,802  count=  196  freq=1.101%
  Aeneid             tokens= 81,140  count=  772  freq=0.951%
  Georgics+Aeneid    tokens= 98,942  count=  968  freq=0.978%
  Cento (question)   tokens=  1,079  count=    4  freq=0.371%


In [3]:
queries = ["Ausonius", "Cento (question)"]
comparisons = ["Georgics", "Aeneid", "Georgics+Aeneid"]

for word in WORDS:
    print(f"\n=== {word} ===")
    for query in queries:
        q_total, q_count, q_freq = stats[word][query]
        for comp in comparisons:
            c_total, c_count, c_freq = stats[word][comp]
            alt = "less" if q_freq < c_freq else "greater"
            table = [[q_count, q_total - q_count], [c_count, c_total - c_count]]
            or_, p = fisher_exact(table, alternative=alt)
            print(f"  {query:<18} vs {comp:<18}  OR={or_:0.4f}  "
                  f"p({alt})={p:.6g}")


=== et ===
  Ausonius           vs Georgics            OR=0.6765  p(less)=2.20124e-09
  Ausonius           vs Aeneid              OR=0.8149  p(less)=0.000101381
  Ausonius           vs Georgics+Aeneid     OR=0.7861  p(less)=3.83099e-06
  Cento (question)   vs Georgics            OR=0.6621  p(less)=0.0279391
  Cento (question)   vs Aeneid              OR=0.7975  p(less)=0.164975
  Cento (question)   vs Georgics+Aeneid     OR=0.7693  p(less)=0.122935

=== in ===
  Ausonius           vs Georgics            OR=0.7542  p(less)=0.00560408
  Ausonius           vs Aeneid              OR=0.8740  p(less)=0.0717642
  Ausonius           vs Georgics+Aeneid     OR=0.8498  p(less)=0.0344048
  Cento (question)   vs Georgics            OR=0.3342  p(less)=0.00941587
  Cento (question)   vs Aeneid              OR=0.3874  p(less)=0.0247914
  Cento (question)   vs Georgics+Aeneid     OR=0.3766  p(less)=0.0203681
